# Julia notebook to compute the symbolic quadratures for the solution of the robust diagnostic PG problem
twnh 

Aug '26.  

This notebook reads the output from `BuoyancyEquationSolution.ipynb` and `FrictionalGeostrophicEquationSolution.ipynb`, namely the Green's functions $G_{\mathfrak{u}}$ and $G_b$ and computes integrals using them. It then writes out the following expression for later use.

\begin{align}
G_{ \mathfrak{u}, b} (z; \xi') & = 
\int_{-H}^{0} G_{ \mathfrak{u}} (z; \xi) \, \int_{\xi}^{0}  G_b(z'; \xi') \; dz'  \; d \xi .
\end{align}

See `SymbolicsTesting.jl` to test this result against numerical quadrature.

The notebook makes a plot of this $G_{ \mathfrak{u}, b} (z; \xi')$ function.

In [1]:
using SymPy
using Symbolics
using JLD2

In [2]:
# Geometry symbolic parameters:
z, ξ   = symbols("z ξ",   real=true, negative=true) # Vertical coordinate and source location (both in [-H,0])
x, y   = symbols("x y",   real=true)                # Horizontal coordinates
H      = symbols("H",   real=true, positive=true)   # Domain depth

ψ      = symbols("ψ",     real=true, positive=true) 
φ      = symbols("φ", imaginary=true)

κ      = symbols("κ",     real=true, positive=true)  # Diffusivity parameter
ν      = symbols("ν",     real=true, positive=true)  # Viscosity parameters

# Dictionary for parsing strings.
locals = Dict(
    "x" => x,
    "y" => y,
    "z" => z,
    "ξ" => ξ,
    "κ" => κ,
    "ν" => ν,
    "φ" => φ,
    "ψ" => ψ,
    "H" => H
    )

# Parse expressions using the canonical symbols.
Guv_str = read("sympy_Guv_expression.txt", String)
Guv = sympy.sympify(Guv_str, locals=locals)
println("Guv expression loaded from file:")
display(Guv)
println()

Gb_str  = read("sympy_Gb_expression.txt", String)
Gb  = sympy.sympify(Gb_str,  locals=locals)
println("Gb expression loaded from file:")
display(Gb)
println()

Guv expression loaded from file:

Gb expression loaded from file:



⎧⎛ 2⋅ξ⋅φ    2⋅φ⋅(H + z)    2⋅φ⋅(H + z + ξ)    ⎞  -φ⋅(z + ξ)           
⎪⎝ℯ      - ℯ            - ℯ                + 1⎠⋅ℯ                     
⎪──────────────────────────────────────────────────────────  for z ≤ ξ
⎪                          ⎛ 2⋅H⋅φ    ⎞                               
⎪                    2⋅ν⋅φ⋅⎝ℯ      + 1⎠                               
⎨                                                                     
⎪⎛ 2⋅z⋅φ    2⋅φ⋅(H + ξ)    2⋅φ⋅(H + z + ξ)    ⎞  -φ⋅(z + ξ)           
⎪⎝ℯ      - ℯ            - ℯ                + 1⎠⋅ℯ                     
⎪──────────────────────────────────────────────────────────  for z ≥ ξ
⎪                          ⎛ 2⋅H⋅φ    ⎞                               
⎩                    2⋅ν⋅φ⋅⎝ℯ      + 1⎠                               

⎧⎛     -2⋅ξ⋅ψ⎞ ⎛ 2⋅H⋅ψ    -2⋅z⋅ψ⎞  ψ⋅(z + ξ)           
⎪⎝1 - ℯ      ⎠⋅⎝ℯ      + ℯ      ⎠⋅ℯ                    
⎪───────────────────────────────────────────  for z ≤ ξ
⎪                  ⎛ 2⋅H⋅ψ    ⎞                        
⎪            2⋅κ⋅ψ⋅⎝ℯ      + 1⎠                        
⎨                                                      
⎪⎛     -2⋅z⋅ψ⎞ ⎛ 2⋅H⋅ψ    -2⋅ξ⋅ψ⎞  ψ⋅(z + ξ)           
⎪⎝1 - ℯ      ⎠⋅⎝ℯ      + ℯ      ⎠⋅ℯ                    
⎪───────────────────────────────────────────  for z ≥ ξ
⎪                  ⎛ 2⋅H⋅ψ    ⎞                        
⎩            2⋅κ⋅ψ⋅⎝ℯ      + 1⎠                        

In [3]:
@time begin
    zp = symbols("zp",     real=true, negative=true)
    ξp = symbols("ξp",     real=true, negative=true)
    tmp_Gb = sympy.expand(Gb.subs(Dict(z => zp, ξ => ξp)))
    inner_Gb_integral = sympy.integrate(tmp_Gb, (zp, ξ, 0))
    max_obj = sympy.Max(ξ,ξp)
    min_obj = sympy.Min(0, max_obj)
    inner_Gb_integral = inner_Gb_integral.subs(min_obj,max_obj)
    # Create two explicit branches
    # expr_if_xi_ge_xip = sympy.expand(inner_Gb_integral.subs(sympy.Max(ξ, ξp), ξ))
    # expr_if_xi_lt_xip = sympy.expand(inner_Gb_integral.subs(sympy.Max(ξ, ξp),ξp))
    expr_if_xi_ge_xip = inner_Gb_integral.subs(sympy.Max(ξ, ξp), ξ)
    expr_if_xi_lt_xip = inner_Gb_integral.subs(sympy.Max(ξ, ξp),ξp)

    # Form a piecewise object
    inner_Gb_integral = sympy.Piecewise(
    (expr_if_xi_ge_xip, sympy.Ge(ξ, ξp)),
    (expr_if_xi_lt_xip, sympy.Lt(ξ, ξp)))
end

println("1.1 Symbolic computation of the integral of Gb completed.")
display(inner_Gb_integral)

  0.518593 seconds (661.81 k allocations: 32.125 MiB, 2.49% gc time, 26.36% compilation time)
1.1 Symbolic computation of the integral of Gb completed.


⎧                                                                              ↪
⎪                                                                              ↪
⎪                                                                              ↪
⎪                                                                              ↪
⎪                                                                              ↪
⎨                                                                              ↪
⎪  ⎛       2  2⋅H⋅ψ  -ξp⋅ψ        2  2⋅H⋅ψ  -3⋅ξp⋅ψ        2  -ξp⋅ψ        2   ↪
⎪  ⎝- 2⋅κ⋅ψ ⋅ℯ     ⋅ℯ      + 2⋅κ⋅ψ ⋅ℯ     ⋅ℯ        - 2⋅κ⋅ψ ⋅ℯ      + 2⋅κ⋅ψ ⋅ℯ ↪
⎪- ─────────────────────────────────────────────────────────────────────────── ↪
⎪                                                                   2  4  4⋅H⋅ ↪
⎩                                                                4⋅κ ⋅ψ ⋅ℯ     ↪

↪                                                                              ↪
↪                          

In [ ]:
@time begin
    # ------------------------------------------------------------------
    # Extract branch expressions
    # Guv branches:
    #   branch 1: for z ≤ ξ
    #   branch 2: for z ≥ ξ
    # inner_Gb_integral branches:
    #   branch 1: for ξ ≥ ξp
    #   branch 2: for ξ < ξp
    # ------------------------------------------------------------------
    Gu_branch1 = Guv.args[1].args[1]                    # z ≤ ξ
    Gu_branch2 = Guv.args[2].args[1]                    # z ≥ ξ

    Gbint_branch1 = inner_Gb_integral.args[1].args[1]   # ξ ≥ ξp
    Gbint_branch2 = inner_Gb_integral.args[2].args[1]   # ξ < ξp

    # ------------------------------------------------------------------
    # Build the four branch products
    # ------------------------------------------------------------------
    f11 = sympy.simplify(Gu_branch1 * Gbint_branch1)  # z ≤ ξ, ξ ≥ ξp
    f12 = sympy.simplify(Gu_branch1 * Gbint_branch2)  # z ≤ ξ, ξ <  ξp
    f21 = sympy.simplify(Gu_branch2 * Gbint_branch1)  # z ≥ ξ, ξ ≥ ξp
    f22 = sympy.simplify(Gu_branch2 * Gbint_branch2)  # z ≥ ξ, ξ <  ξp

    # ------------------------------------------------------------------
    # Case A: z ≤ ξp
    #
    # Interval split:
    #   ξ ∈ [-H, z]     -> Gu branch 1, inner branch 1
    #   ξ ∈ [z, ξp]     -> Gu branch 2, inner branch 1
    #   ξ ∈ [ξp, 0]     -> Gu branch 2, inner branch 2
    # ------------------------------------------------------------------
    I_A_11 = sympy.integrate(f11, (ξ,-H,  z))
    I_A_21 = sympy.integrate(f21, (ξ, z, ξp))
    I_A_22 = sympy.integrate(f22, (ξ,ξp,  0))
    I_A = sympy.simplify(I_A_11 + I_A_21 + I_A_22)

    # ------------------------------------------------------------------
    # Case B: ξp < z
    #
    # Interval split:
    #   ξ ∈ [-H, ξp]    -> Gu branch 1, inner branch 1
    #   ξ ∈ [ξp, z]     -> Gu branch 1, inner branch 2
    #   ξ ∈ [z, 0]      -> Gu branch 2, inner branch 2
    # ------------------------------------------------------------------
    I_B_11 = sympy.integrate(f11, (ξ,-H, ξp))
    I_B_12 = sympy.integrate(f12, (ξ, ξp, z)) 
    I_B_22 = sympy.integrate(f22, (ξ,  z, 0))
    I_B = sympy.simplify(I_B_11 + I_B_12 + I_B_22)

    # ------------------------------------------------------------------
    # Final piecewise outer integral
    # ------------------------------------------------------------------
   Guvb = sympy.Piecewise(
        (sympy.factor(I_A), sympy.Le(z, ξp)),
        (sympy.factor(I_B), sympy.Gt(z, ξp))
    )
end

println("1.2 Symbolic computation of the integral of G_u and Gb completed.")
display(Guvb)

 99.575435 seconds (148.68 k allocations: 5.560 MiB, 0.02% compilation time)
1.2 Symbolic computation of the integral of G_u and Gb completed.


⎧                                                                              ↪
⎪                                                                              ↪
⎪                                                                              ↪
⎪                                                                              ↪
⎪                                                                              ↪
⎨                                                                              ↪
⎪⎛ 2  4⋅H⋅φ  4⋅H⋅ψ  2⋅z⋅φ  z⋅ψ  ξp⋅φ  2⋅ξp⋅ψ      2  4⋅H⋅φ  3⋅H⋅ψ  2⋅z⋅φ  z⋅ψ  ↪
⎪⎝φ ⋅ℯ     ⋅ℯ     ⋅ℯ     ⋅ℯ   ⋅ℯ    ⋅ℯ       - 2⋅φ ⋅ℯ     ⋅ℯ     ⋅ℯ     ⋅ℯ   ⋅ ↪
⎪───────────────────────────────────────────────────────────────────────────── ↪
⎪                                                                              ↪
⎩                                                                              ↪

↪                                                                              ↪
↪                          

In [ ]:
function sympy_piecewise_to_symbolics(P, condsym)
    branches = P.args
    expr1 = sympy_to_symbolics(branches[1].args[1], [])
    expr2 = sympy_to_symbolics(branches[2].args[1], [])
    return ifelse(condsym, expr1, expr2)
end

@variables z ξ ξp
symbolics_Guvb = sympy_piecewise_to_symbolics(Guvb, z <= ξp)
symbolics_inner_Gb_integral = sympy_piecewise_to_symbolics(inner_Gb_integral, ξ >= ξp)
symbolics_Gb   = sympy_piecewise_to_symbolics(Gb, z <= ξ)
symbolics_Guv  = sympy_piecewise_to_symbolics(Guv, z <= ξ)

@save "symexpr.jld2" symbolics_Guvb symbolics_Gb symbolics_Guv symbolics_inner_Gb_integral
println("1.3 Expression converted to Symbolics and saved to file: symexpr.jld2")

1.3 Expression converted to Symbolics and saved to file: symexpr.jld2


#### Make plots

In [8]:
using CairoMakie

CairoMakie.activate!()

# -------------------------------------------------------------------
# Parameters
# -------------------------------------------------------------------
Hval = 1.0
Ek1 = 0.25
Ek2 = 0.025
Ek3 = 0.0025
# Ek_vals = [Ek1, Ek2, Ek3]
Ek_vals = [Ek1, Ek2]
varϕ_vals = [sqrt(1im / Ek) for Ek in Ek_vals]
ψ_vals = [sqrt(1 / Ek) for Ek in Ek_vals]
zgrid = range(-Hval*0.999, 0.0, length=128)   # avoids undefined phase angle at z=-H
ξp_vals = range(-Hval, 0, length=5)

# Colors for ξ families
ξcols = cgrad(:viridis, length(ξp_vals), categorical=true)

# -------------------------------------------------------------------
# Publication-style theme
# -------------------------------------------------------------------
set_theme!(
    Theme(
        fontsize=16,
        fonts=(; regular="TeX Gyre Heros"),
        Axis=(
            titlesize=20,
            xlabelsize=18,
            ylabelsize=18,
            xticklabelsize=16,
            yticklabelsize=16,
            xgridvisible=true,
            ygridvisible=true,
            xgridstyle=:dash,
            ygridstyle=:dash,
            xgridcolor=(:gray70, 0.35),
            ygridcolor=(:gray70, 0.35),
            spinewidth=1.2,
            xticksize=5,
            yticksize=5,
            xminorticksvisible=false,
            yminorticksvisible=false,
            topspinevisible=true,
            rightspinevisible=true,
        ),
        Legend=(
            framevisible=false,
            labelsize=15,
            titlesize=16,
            patchsize=(22, 14),
        ),
        Lines=(
            linewidth=2.5,
        )
    )
)

# -------------------------------------------------------------------
# Styling helper
# -------------------------------------------------------------------
function style_axis!(ax)
    ax.topspinevisible = true
    ax.rightspinevisible = true
    ax.xgridvisible = true
    ax.ygridvisible = true
    ax.xgridstyle = :dash
    ax.ygridstyle = :dash
    ax.xgridcolor = (:gray70, 0.35)
    ax.ygridcolor = (:gray70, 0.35)
    return ax
end

# Colormap that fades from white to a given color
function fade_cmap(c)
    return cgrad([RGBAf(1, 1, 1, 0.12), c])
end

# -------------------------------------------------------------------
# Figure layout
# -------------------------------------------------------------------
fig = Figure(size=(1200, 1450))

Label(fig[0, 1:2],
    "Integrated Green's function kernel: magnitude and phase",
    fontsize=22,
    font=:bold
)

# -------------------------------------------------------------------
# Helper to plot one Ek row
# -------------------------------------------------------------------
function plot_ek_row!(fig, row, Ek, varϕ, ψ; Hval=Hval, zgrid=zgrid)

    if row ==  1
        panel_labels = ("(a) ", "(b) ")
    elseif row ==2
        panel_labels = ("(c) ", "(d) ")
    elseif row == 3
        panel_labels = ("(e) ", "(f) ")
    end # if

    ax_mag = Axis(fig[row, 1],
        xlabel="magnitude",
        ylabel="depth z/H",
        title="$(panel_labels[1])Ek = $(Ek): magnitude"
    )

    ax_ph = Axis(fig[row, 2],
        xlabel="phase [degrees]",
        ylabel="depth z/H",
        title="$(panel_labels[2])Ek = $(Ek): phase"
    )

    xlims!(ax_ph, -180, 180)

     # ----------------------------------------------------------------
    # G_uvb(z; ξp) family
    # ----------------------------------------------------------------
    for (j, ξp) in enumerate(ξp_vals)
        c = ξcols[j]
        Gvals = [Guvb(Hval, z, ξp,  varϕ, ψ, 1.0, 1.0) for z in zgrid]

        mags = abs.(Gvals)
        phases = rad2deg.(angle.(Gvals))

        lines!(ax_mag, mags, zgrid,
            color=c,
            linewidth=2.4,
            label="G_u, ξ/H = $(round(ξp/Hval, digits=3))"
        )

        if isapprox(ξ, -H; atol=1e-12)
            phases = NaN .* phases
        end

        lines!(ax_ph, phases, zgrid,
            color=mags,
            colormap=fade_cmap(c),
            colorrange=(0, maximum(mags)),
            linewidth=2.4,
            label="G_u, ξ/H = $(round(ξp/Hval, digits=3))"
        )
    end

    return ax_mag, ax_ph
end

# -------------------------------------------------------------------
# Plot both Ek rows
# -------------------------------------------------------------------
ax11, ax12 = plot_ek_row!(fig, 1, Ek_vals[1], varϕ_vals[1], ψ_vals[1]; Hval=Hval)
ax21, ax22 = plot_ek_row!(fig, 2, Ek_vals[2], varϕ_vals[2], ψ_vals[2]; Hval=Hval)
# ax31, ax32 = plot_ek_row!(fig, 3, Ek_vals[3], varϕ_vals[3], ψ_vals[3]; Hval=Hval)

# Link y axes
linkyaxes!(ax11, ax21)
linkyaxes!(ax12, ax22)
# linkyaxes!(ax21, ax31)
# linkyaxes!(ax22, ax32)

# -------------------------------------------------------------------
# Single shared legend below the panels
# -------------------------------------------------------------------
legend_elements = LineElement[]
legend_labels = String[]

for (j, ξp) in enumerate(ξp_vals)
    push!(legend_elements, LineElement(color=ξcols[j], linewidth=3))
    push!(legend_labels, "ξ/H = $(round(ξp/Hval, digits=2))")
end

legend_g = Legend(fig[3, 1:2],
    legend_elements,
    legend_labels,
    title="Green's function panel",
    orientation=:horizontal,
    nbanks=2,
    framevisible=false,
    tellwidth=false,
    halign=:center
)

# -------------------------------------------------------------------
# Tightened spacing
# -------------------------------------------------------------------
colgap!(fig.layout, 1, 10)
rowgap!(fig.layout, 1, 4)
rowgap!(fig.layout, 2, 2)
# rowgap!(fig.layout, 3, 2)

# -------------------------------------------------------------------
# Save to PDF
# -------------------------------------------------------------------
CairoMakie.save("Guvb_magnitude_phase_two_Ek.pdf", fig)

fig

UndefVarError: UndefVarError: `Guvb` not defined in `Main`
Suggestion: check for spelling errors or missing imports.